In [195]:
import pandas as pd
import seaborn as sns 
import matplotlib.pyplot as plt



In [ ]:
#read excelfile out of folder 'data', place there your path to your files, convert excel to .csv. Example for converting:
#df3 = pd.read_excel('data/LFB Mobilisation data from 2015 - 2020.xlsx')
#df3.to_csv('data/LFB_mobilisation_data_2015_2020.csv', index=False)

In [ ]:
#load incident Data
inc_meta = pd.read_csv('data/Incident_metadata.csv')
inc_1 = pd.read_csv('data/Incident_data_2009_2017.csv')
inc_2 = pd.read_csv('data/Incident_data_2018_2023.csv')
inc_3 = pd.read_csv('data/Incident_data_2024_onwards.csv')

inc_all = pd.concat([inc_1, inc_2, inc_3], ignore_index=True)


/var/folders/ml/mylzx0q15zj5ydny113g9v_80000gn/T/ipykernel_95897/586915367.py:3: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  inc_1 = pd.read_csv('data/Incident_data_2009_2017.csv')


In [ ]:
#load mobilisation_data
mob_meta = pd.read_csv('data/Mobilisation_metadata.csv')
mob_1 = pd.read_csv('data/Mobilisation_data_2009_2014.csv')
mob_2 = pd.read_csv('data/Mobilisation_data_2015_2020.csv')
mob_3 = pd.read_csv('data/Mobilisation_data_2021_2024.csv')
mob_4 = pd.read_csv('data/Mobilisation_data_2025.csv')
mob_all = pd.concat([mob_1, mob_2, mob_3, mob_4], ignore_index=True)

/var/folders/ml/mylzx0q15zj5ydny113g9v_80000gn/T/ipykernel_95897/582136193.py:4: DtypeWarning: Columns (0,13) have mixed types. Specify dtype option on import or set low_memory=False.
  mob_2 = pd.read_csv('data/Mobilisation_data_2015_2020.csv')
/var/folders/ml/mylzx0q15zj5ydny113g9v_80000gn/T/ipykernel_95897/582136193.py:6: DtypeWarning: Columns (15) have mixed types. Specify dtype option on import or set low_memory=False.
  mob_4 = pd.read_csv('data/Mobilisation_data_2025.csv')


In [197]:
inc_all.to_csv('data/Incident_data_combined_2009_2026.csv', index=False)
mob_all.to_csv('data/Mobilisation_data_combined_2009_2026.csv', index=False)

In [190]:
#missing data for all columns in prozent

missing_data_mob_all = mob_all.isna().mean() * 100
missing_data_mob_all
#or 
# missing_values_mob = (mob_all.isna().sum() / len(mob_all)) * 100
#print(missing_values_mob)

missing_data_inc_all = inc_all.isna().mean() * 100
missing_data_inc_all

IncidentNumber                             0.000000
DateOfCall                                 0.000000
CalYear                                    0.000000
TimeOfCall                                 0.000000
HourOfCall                                 0.000000
IncidentGroup                              0.002766
StopCodeDescription                        0.000000
SpecialServiceType                        66.674827
PropertyCategory                           0.000563
PropertyType                               0.000563
AddressQualifier                           0.000051
Postcode_full                             51.221371
Postcode_district                          0.000000
UPRN                                       7.238344
USRN                                       8.341408
IncGeo_BoroughCode                         0.000000
IncGeo_BoroughName                         0.000000
ProperCase                                 0.000000
IncGeo_WardCode                            0.038053
IncGeo_WardN

In [ ]:
# Group by code and list the unique names
ward_check = inc_all.groupby('IncGeo_WardCode')['IncGeo_WardName'].nunique()

# Only show those with more than one name
print(ward_check[ward_check > 1])

# show word pairs
ward_pairs = inc_all.groupby('IncGeo_WardCode')['IncGeo_WardName'].unique()
print(ward_pairs[ward_pairs.apply(len) > 1].head(10))

IncGeo_WardCode
E05000028    2
E05000029    2
E05000035    2
E05000081    2
E05000082    2
            ..
E05014115    2
E05014116    2
E05014117    2
E05014118    2
E05014119    2
Name: IncGeo_WardName, Length: 753, dtype: int64
IncGeo_WardCode
E05000028                  [Becontree, BECONTREE]
E05000029        [Chadwell Heath, CHADWELL HEATH]
E05000035                [Longbridge, LONGBRIDGE]
E05000081                 [St. Mary's, St Mary's]
E05000082           [St. Michael's, St Michael's]
E05000092            [Kensal Green, KENSAL GREEN]
E05000111              [Chislehurst, CHISLEHURST]
E05000114    [Cray Valley East, CRAY VALLEY EAST]
E05000116        [Crystal Palace, CRYSTAL PALACE]
E05000127            [West Wickham, WEST WICKHAM]
Name: IncGeo_WardName, dtype: object


In [ ]:
# All Incident-Nummern from incident DataFrame 
inc_numbers = set(inc_all['IncidentNumber'])

# Filter the mobilisatation Incident ID who are not in there
mobs_without_inc = mob_all[~mob_all['IncidentNumber'].isin(inc_numbers)]

# Jetzt schau dir an, welche Bezirke dort stehen
print("Boroughs in Mobilisation without Data in Incident:")
print(mobs_without_inc['BoroughName'].value_counts())

In [ ]:
print(mobs_without_inc.nunique())
print(len(mobs_without_inc))

In [200]:
inc_all.duplicated().sum()

np.int64(0)

In [203]:
mob_all['DeployedFromStation_Name'].nunique()

128

In [204]:
address = pd.read_csv('data/station_addresses.csv')

In [205]:
address.head()

,Station Name,Address
0,Barking,"Alfred's Way, IG11 0BB"
1,Dagenham,"70 Rainham Road North, Dagenham, RM10 7ES"
2,Dowgate,"94-95 Upper Thames Street, EC4R 3UE"
3,Shoreditch,"235 Old Street, EC1V 9EY"
4,Stoke Newington,"64 Stoke Newington Church Street, N16 0AR"


In [206]:
namen_aus_mob = mob_all['DeployedFromStation_Name'].unique()

# 2. Prüfen, welche davon NICHT in address['Station Name'] enthalten sind
# Das ~ Symbol kehrt das isin() um (bedeutet: "ist nicht in")
fehlt_in_address = [name for name in namen_aus_mob if name not in address['Station Name'].values]

In [ ]:
print(fehlt_in_address)
print(len(fehlt_in_address))


['Clerkenwell', 'Southwark', 'Eltham', 'Hammersmith', 'Belsize', 'Woolwich', 'Silvertown', 'Knightsbridge', 'Westminster', 'Kingsland', 'Bow', 'Downham', 'Grays', 'Epsom', 'Cheshunt', 'Godstone', nan, 'Dartford', 'Esher', 'Staines', 'Watford', 'Kent', 'Fordbridge', 'Hertfordshire', 'Surrey', 'Essex', 'Buckinghamshire', 'Loughton', 'Borehamwood']
29


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 670635 entries, 0 to 670634
Data columns (total 39 columns):
 #   Column                                  Non-Null Count   Dtype  
---  ------                                  --------------   -----  
 0   IncidentNumber                          670635 non-null  object 
 1   DateOfCall                              670635 non-null  object 
 2   CalYear                                 670635 non-null  int64  
 3   TimeOfCall                              670635 non-null  object 
 4   HourOfCall                              670635 non-null  int64  
 5   IncidentGroup                           670635 non-null  object 
 6   StopCodeDescription                     670635 non-null  object 
 7   SpecialServiceType                      232510 non-null  object 
 8   PropertyCategory                        670635 non-null  object 
 9   PropertyType                            670635 non-null  object 
 10  AddressQualifier                        6706

,IncidentNumber,DateOfCall,CalYear,TimeOfCall,HourOfCall,IncidentGroup,StopCodeDescription,SpecialServiceType,PropertyCategory,PropertyType,...,FirstPumpArriving_AttendanceTime,FirstPumpArriving_DeployedFromStation,SecondPumpArriving_AttendanceTime,SecondPumpArriving_DeployedFromStation,NumStationsWithPumpsAttending,NumPumpsAttending,PumpCount,PumpMinutesRounded,Notional Cost (£),NumCalls
0,000008-01012018,2018-01-01,2018,00:04:25,0,False Alarm,AFA,NaN,Non Residential,Mosque,...,348.0,Finchley,NaN,NaN,1.0,1.0,1,60,328,1.0
1,000009-01012018,2018-01-01,2018,00:04:30,0,False Alarm,AFA,NaN,Non Residential,Pub/wine bar/bar,...,144.0,Beckenham,NaN,NaN,1.0,1.0,1,60,328,1.0
2,000010-01012018,2018-01-01,2018,00:04:34,0,Fire,Secondary Fire,NaN,Outdoor Structure,Common external bin storage area,...,232.0,Southgate,NaN,NaN,1.0,1.0,1,60,328,1.0
3,000011-01012018,2018-01-01,2018,00:04:58,0,Special Service,Special Service,RTC,Road Vehicle,Multiple Vehicles,...,22.0,Enfield,NaN,NaN,1.0,1.0,1,60,328,1.0
4,000014-01012018,2018-01-01,2018,00:07:47,0,Fire,Primary Fire,NaN,Road Vehicle,Car,...,241.0,Stratford,NaN,NaN,1.0,1.0,1,60,328,6.0
